# TCGA-BRCA: Histopathology Image Quality Assessment (IQA) & Analytics
## Google Colab Execution Notebook
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Youssef-Ebn-Yasser/breastCancerAnalytics/blob/main/TCGA-BRCA/tcga_brca_colab.ipynb)

### Modality: Breast Histopathology (Whole Slide Imaging & Diagnostic Patches)
### Dataset: The Cancer Genome Atlas Breast Invasive Carcinoma (TCGA-BRCA) / Breast Histopathology Images

---

## 📌 Colab Workflow Overview
1. **Colab Environment Setup:** Auto-installs prerequisites and mounts directories.
2. **Dataset Acquisition (3 Easy Options):**
   - **Option A (Kaggle API):** Fast direct download using your Kaggle credentials.
   - **Option B (Google Drive):** Mount your Google Drive folder directly.
   - **Option C (Self-Contained Fallback):** Automatically generates sample histology patches so the notebook runs instantly without manual downloads.
3. **Dataset Characterization:** Analyzes diagnostic classes (IDC Malignant vs Normal Benign) and preparation types (FFPE vs Frozen Section).
4. **Image Quality Assessment (IQA):** Computes Laplacian Variance, Tenengrad Gradient, RMS Contrast, Michelson Contrast, Shannon Entropy, and Blind SNR (MAD).
5. **Statistical Analysis & Visual Montages:** Generates 4 publication-grade figures.
6. **One-Click Download:** Packages all generated figures and summary CSVs into a zip file and triggers automatic browser download to your local machine!


In [ ]:
# Cell 1: Environment Setup & Library Installation
import os
import sys
import glob
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from pathlib import Path
from scipy.stats import entropy

# Set high-resolution plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.autolayout'] = True

OUTPUT_DIR = "/content/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"[INFO] Colab Environment ready. Outputs will be saved to: {OUTPUT_DIR}")


In [ ]:
# Cell 2: Dataset Acquisition (Choose Option A, B, or C)

# ==============================================================================
# OPTION A: Download via Kaggle API in Colab (Fastest if you have kaggle.json)
# Uncomment the block below to use:
# ==============================================================================
# from google.colab import files
# print("Please upload your kaggle.json file:")
# files.upload()
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !mkdir -p /content/dataset
# !kaggle datasets download -d paultimothymooney/breast-histopathology-images -p /content/dataset --unzip

# ==============================================================================
# OPTION B: Mount Google Drive if dataset is in your Drive
# Uncomment the block below to use:
# ==============================================================================
# from google.colab import drive
# drive.mount('/content/drive')
# DRIVE_DATA_PATH = "/content/drive/MyDrive/breast_cancer_data"

# ==============================================================================
# OPTION C: Automatic Path Detection & Self-Contained Fallback
# ==============================================================================
CANDIDATE_DIRS = [
    "/content/dataset",
    "/content/drive/MyDrive/breast_cancer_data",
    "/content/drive/MyDrive/tcga-brca",
    "/content/sample_data/tcga",
]

def locate_or_create_dataset():
    for p in CANDIDATE_DIRS:
        if os.path.exists(p) and len(os.listdir(p)) > 0:
            print(f"[SUCCESS] Found existing dataset at: {p}")
            return p
    
    print("[INFO] No external dataset detected. Generating self-contained histology cohort...")
    target_dir = "/content/sample_data/tcga"
    os.makedirs(target_dir, exist_ok=True)
    classes = ["IDC_Malignant", "ILC_Malignant", "Normal_Benign"]
    preparations = ["FFPE", "Frozen_Section"]
    np.random.seed(42)

    for i in range(60):
        cls = classes[i % len(classes)]
        prep = preparations[(i // len(classes)) % len(preparations)]
        cls_dir = os.path.join(target_dir, cls)
        os.makedirs(cls_dir, exist_ok=True)

        h, w = 256, 256
        img = np.ones((h, w, 3), dtype=np.uint8)
        img[:, :, 0] = np.random.randint(200, 230)
        img[:, :, 1] = np.random.randint(170, 200)
        img[:, :, 2] = np.random.randint(220, 245)

        for _ in range(25):
            pt1 = (np.random.randint(0, w), np.random.randint(0, h))
            pt2 = (np.random.randint(0, w), np.random.randint(0, h))
            cv2.line(img, pt1, pt2, (180, 140, 210), 1)

        n_nuclei = 120 if cls == "IDC_Malignant" else (60 if cls == "ILC_Malignant" else 30)
        for _ in range(n_nuclei):
            cx = np.random.randint(15, w - 15)
            cy = np.random.randint(15, h - 15)
            r = np.random.randint(3, 7)
            cv2.circle(img, (cx, cy), r, (130, 40, 80), -1)

        if prep == "Frozen_Section":
            k = np.random.choice([3, 5, 7])
            img = cv2.GaussianBlur(img, (k, k), 0)
            if np.random.rand() > 0.5:
                cv2.line(img, (0, np.random.randint(20, h - 20)), (w, np.random.randint(20, h - 20)), (240, 240, 240), 2)

        cv2.imwrite(os.path.join(cls_dir, f"TCGA_BRCA_{prep}_{cls}_{i:03d}.png"), img)
    return target_dir

DATASET_ROOT = locate_or_create_dataset()
print(f"[ACTIVE DATASET DIRECTORY] {DATASET_ROOT}")


## 1. Dataset Characterization & Metadata Loading

In [ ]:
# Cell 3: Load Metadata & Quantify Class Distribution
image_extensions = ("*.png", "*.jpg", "*.jpeg", "*.tif", "*.tiff")
all_paths = []
for ext in image_extensions:
    all_paths.extend(glob.glob(os.path.join(DATASET_ROOT, "**", ext), recursive=True))

records = []
for path in all_paths:
    fname = Path(path).name
    parent = Path(path).parent.name
    prep = "Frozen_Section" if ("frozen" in fname.lower() or "frozen" in parent.lower()) else "FFPE"
    
    label = "Normal_Benign"
    if "idc" in fname.lower() or "idc" in parent.lower():
        label = "IDC_Malignant"
    elif "ilc" in fname.lower() or "ilc" in parent.lower():
        label = "ILC_Malignant"
    elif "malignant" in fname.lower() or "malignant" in parent.lower():
        label = "IDC_Malignant"

    records.append({
        "image_id": fname,
        "file_path": path,
        "ground_truth_label": label,
        "preparation_type": prep
    })

df_meta = pd.DataFrame(records)
print(f"Total Cohort Images Available: {len(df_meta)}")
print("\n--- Class Distribution ---")
print(df_meta["ground_truth_label"].value_counts())
print("\n--- Slide Preparation Formats ---")
print(df_meta["preparation_type"].value_counts())


## 2. Image Quality Assessment (IQA) Formulations

In [ ]:
# Cell 4: Define Mathematical Quality Indicators
def compute_laplacian_variance(gray):
    """Measures edge crispness vs. optical defocus blur."""
    return float(np.var(cv2.Laplacian(gray, cv2.CV_64F)))

def compute_tenengrad(gray):
    """Measures Sobel gradient energy along cellular boundaries."""
    gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    return float(np.mean(gx**2 + gy**2))

def compute_rms_contrast(gray):
    """Standard deviation of normalized luminance [0, 1]."""
    norm = gray.astype(np.float64) / 255.0
    return float(np.std(norm))

def compute_michelson_contrast(gray):
    """Dynamic luminance spread."""
    p1 = float(np.percentile(gray, 1))
    p99 = float(np.percentile(gray, 99))
    return float((p99 - p1) / (p99 + p1 + 1e-6))

def compute_shannon_entropy(gray):
    """Information content & textural richness in bits."""
    hist, _ = np.histogram(gray.ravel(), bins=256, range=(0, 256), density=True)
    hist = hist[hist > 0]
    return float(entropy(hist, base=2))

def compute_snr_mad(gray):
    """Blind SNR (dB) using high-frequency Median Absolute Deviation."""
    mean_val = float(np.mean(gray))
    lap = cv2.Laplacian(gray, cv2.CV_64F, ksize=3)
    mad = float(np.median(np.abs(lap - np.median(lap))))
    sigma_n = mad / 0.6745
    if sigma_n < 1e-6:
        return 45.0
    snr = mean_val / (sigma_n + 1e-6)
    return float(np.clip(20.0 * np.log10(max(snr, 1e-6)), -10.0, 50.0))

def extract_histology_iqa(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return None
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    lap = compute_laplacian_variance(gray)
    ten = compute_tenengrad(gray)
    rms = compute_rms_contrast(gray)
    mic = compute_michelson_contrast(gray)
    ent = compute_shannon_entropy(gray)
    snr = compute_snr_mad(gray)
    
    # Normalized Composite Quality Score [0.0 to 1.0]
    norm_lap = np.clip(np.log1p(lap) / 8.0, 0.0, 1.0)
    norm_contrast = np.clip(rms * 4.0, 0.0, 1.0)
    norm_snr = np.clip((snr - 5.0) / 25.0, 0.0, 1.0)
    norm_ent = np.clip(ent / 8.0, 0.0, 1.0)
    composite = float(0.30 * norm_lap + 0.25 * norm_contrast + 0.25 * norm_snr + 0.20 * norm_ent)
    
    tier = "High_Quality" if composite >= 0.65 else ("Medium_Quality" if composite >= 0.45 else "Degraded")
    
    return {
        "laplacian_variance": lap,
        "tenengrad_gradient": ten,
        "rms_contrast": rms,
        "michelson_contrast": mic,
        "shannon_entropy": ent,
        "snr_db": snr,
        "composite_quality_score": composite,
        "quality_tier": tier
    }

print("[INFO] IQA mathematical functions compiled successfully.")


In [ ]:
# Cell 5: Execute IQA Feature Extraction
# Evaluate up to 300 images for fast, representative execution in Colab
sample_size = min(len(df_meta), 300)
eval_df = df_meta.head(sample_size).copy()
print(f"[INFO] Computing IQA metrics across {sample_size} histology slides...")

iqa_records = []
for idx, row in eval_df.iterrows():
    metrics = extract_histology_iqa(row["file_path"])
    if metrics:
        metrics["image_id"] = row["image_id"]
        iqa_records.append(metrics)

df_iqa = pd.DataFrame(iqa_records)
df_full = pd.merge(eval_df, df_iqa, on="image_id")
print("[SUCCESS] Feature extraction completed.")
df_full[["ground_truth_label", "preparation_type", "laplacian_variance", "rms_contrast", "shannon_entropy", "snr_db", "composite_quality_score"]].head()


## 3. Statistical Analysis & Visual Inspection

In [ ]:
# Cell 6: Generate Statistical Summaries & Figures
print("=" * 65)
print("TCGA-BRCA HISTOPATHOLOGY IQA: DESCRIPTIVE STATISTICS")
print("=" * 65)
stats_summary = df_full[["laplacian_variance", "tenengrad_gradient", "rms_contrast", "shannon_entropy", "snr_db", "composite_quality_score"]].describe()
display(stats_summary)
stats_summary.to_csv(os.path.join(OUTPUT_DIR, "tcga_brca_iqa_statistical_summary.csv"))

# 1. Class Distribution Plot
plt.figure(figsize=(8, 4))
sns.countplot(data=df_full, x="ground_truth_label", hue="preparation_type", palette="Set2")
plt.title("TCGA-BRCA: Cohort Class Distribution & Preparation Type", fontsize=12, fontweight="bold")
plt.savefig(os.path.join(OUTPUT_DIR, "01_tcga_class_distribution.png"), dpi=200)
plt.show()

# 2. Quality Distributions
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
sns.histplot(df_full["laplacian_variance"], kde=True, ax=axes[0, 0], color="crimson")
axes[0, 0].set_title("Sharpness (Laplacian Variance)", fontweight="bold")

sns.histplot(df_full["rms_contrast"], kde=True, ax=axes[0, 1], color="teal")
axes[0, 1].set_title("RMS Dynamic Contrast", fontweight="bold")

sns.histplot(df_full["shannon_entropy"], kde=True, ax=axes[1, 0], color="purple")
axes[1, 0].set_title("Shannon Information Entropy (bits)", fontweight="bold")

sns.histplot(df_full["snr_db"], kde=True, ax=axes[1, 1], color="darkorange")
axes[1, 1].set_title("Signal-to-Noise Ratio (dB)", fontweight="bold")
plt.suptitle("TCGA-BRCA: Quality Metric Distributions", fontsize=14, fontweight="bold")
plt.savefig(os.path.join(OUTPUT_DIR, "02_tcga_metric_distributions.png"), dpi=200)
plt.show()

# 3. FFPE vs Frozen Section Sharpness Comparison
plt.figure(figsize=(8, 4))
sns.boxplot(data=df_full, x="preparation_type", y="laplacian_variance", hue="preparation_type", palette="Pastel1", legend=False)
plt.title("Sharpness Comparison: FFPE vs. Frozen Section Artifacts", fontsize=12, fontweight="bold")
plt.savefig(os.path.join(OUTPUT_DIR, "03_ffpe_vs_frozen_sharpness.png"), dpi=200)
plt.show()

# 4. Best vs Degraded Slides Visual Montage
sorted_df = df_full.sort_values(by="composite_quality_score", ascending=False)
best_rows = sorted_df.head(3)
worst_rows = sorted_df.tail(3)

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for i, (_, row) in enumerate(best_rows.iterrows()):
    im = cv2.cvtColor(cv2.imread(row["file_path"]), cv2.COLOR_BGR2RGB)
    axes[0, i].imshow(im)
    axes[0, i].set_title(f"HIGH QUALITY\nScore: {row['composite_quality_score']:.2f} | Lap: {row['laplacian_variance']:.1f}", color="green", fontweight="bold")
    axes[0, i].axis("off")

for i, (_, row) in enumerate(worst_rows.iterrows()):
    im = cv2.cvtColor(cv2.imread(row["file_path"]), cv2.COLOR_BGR2RGB)
    axes[1, i].imshow(im)
    axes[1, i].set_title(f"DEGRADED / BLUR\nScore: {row['composite_quality_score']:.2f} | Lap: {row['laplacian_variance']:.1f}", color="red", fontweight="bold")
    axes[1, i].axis("off")

plt.suptitle("Visual Comparison: High-Fidelity vs Defocused/Artifact Histology Slides", fontsize=14, fontweight="bold")
plt.savefig(os.path.join(OUTPUT_DIR, "04_best_vs_worst_montage.png"), dpi=200)
plt.show()

# Save final enriched metadata CSV
output_csv = os.path.join(OUTPUT_DIR, "tcga_brca_quality_metadata.csv")
df_full.to_csv(output_csv, index=False)
print(f"[SUCCESS] Exported enriched clinical IQA metadata to: {output_csv}")


## 4. One-Click Result Download to Local Machine
Run the cell below to package all generated figures and CSV summary files into a zip archive and trigger an automatic browser download.


In [ ]:
# Cell 7: Archive & Download Results to Local Computer
import shutil

zip_filename = "/content/tcga_brca_colab_results"
shutil.make_archive(zip_filename, 'zip', OUTPUT_DIR)
full_zip_path = f"{zip_filename}.zip"
print(f"[INFO] Archive created: {full_zip_path} ({os.path.getsize(full_zip_path) / 1024:.1f} KB)")

try:
    from google.colab import files
    print("[INFO] Triggering automatic browser download...")
    files.download(full_zip_path)
except ImportError:
    print(f"[NOTICE] Running outside Google Colab. File saved locally at: {full_zip_path}")
